In [1]:
from pathlib import Path
from xlsxwriter import Workbook
import numpy as np
import pandas as pd
import pyodbc
import sys
import os
import time
from datetime import date
import re
import warnings as wr
wr.filterwarnings('ignore')

In [2]:
seasons_as10 = {1:'Kharif', 2:'Phase-I', 3:'Phase-II', 4:'Phase-III', 5:'Summer'}
seasons_as20 = {1:'Kharif', 2:'E.Kharif', 3:'L.Kharif', 4:'Rabi', 5:'Summer'}

In [3]:
def clean_rc123(year_int, season_cd, planning, b4_cod):
    #set database file path
    folder_path = Path('./software')
    for file_path in folder_path.iterdir():
        if file_path.is_file():
            if season_nm.lower() in file_path.name.lower():
                accdb_path = "./" + "/".join(str(file_path).split("\\"))
    
    #create connection with mdb database
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={accdb_path};'
    )
    conn = pyodbc.connect(conn_str)
    
    #get data from the table - RC1
    season_nm = seasons[season_cd]
    st_cd = 1
    state_cd = 33
    year = 2024
    planning = 260
    b4_cod = 250
    season_cd = 2
    ddg = 2510
    ddtrs = 2810
    
    cols1 = ['RC', 'ST', 'YR', 'SESON', 'NOVP', 'NOVCOD', 'STAT', 'DIST', 'STRA',
           'VILL', 'EPC', 'NOVPC', 'NOVTRS', 'CADC', 'TLC', 'MAC', 'MUC', 'DDG',
           'SGC', 'GCC', 'LFOG', 'ROG', 'DDTRS', 'TRSSC', 'TRSSF', 'DCKC', 'HSSN',
           'GEOA', 'TCROPA', 'TNSSN', 'TGEOASN', 'RC61', 'RC62', 'RC63', 'REJ']
    
    sql_query1 = 'SELECT * FROM "Copy Of RC1"'
    df1 = pd.read_sql(sql_query1, conn)[cols1]
    
    df1 = (df1
        .drop_duplicates()
        .apply(pd.to_numeric, errors='coerce')
        .query("STAT == @state_cd and SESON == @season_cd")
        .assign(
            REJ = lambda x : np.where(x['REJ']==1, 1, 0)
            ,NOVPC1 = lambda x : x['NOVPC'].fillna(1)
            ,NOVPC = lambda x : np.where(x['NOVPC1']==0, 1, x['NOVPC1'])
            ,NOVTRS1 = lambda x : x['NOVTRS'].fillna(1)
            ,NOVTRS = lambda x : np.where(x['NOVTRS1']==0, 1, x['NOVTRS1'])
            ,CADC = lambda x : np.where(np.isin(x['CADC'], [1,2,3,4]), x['CADC'], 9)
            ,TLC = lambda x : np.where(x['CADC']==3, 0, np.where(np.isin(x['CADC'], [4,9]), 8, x['TLC']))
            ,MAC = lambda x : np.where(x['MAC']==1, 1, 0)
            ,MUC = lambda x : np.where(x['MAC']==1, x['MUC'], None)
            ,TRSSC = lambda x : np.where(x['TRSSC']==3, 8, x['TRSSC'])
            ,NOVP = planning
            ,DDG = ddg
            ,DDTRS = ddtrs
            ,TCROPA = lambda x : x['TCROPA'].fillna(0)
            ,GEOA = lambda x : x['GEOA'].fillna(0)
            ,YR = year+1
            ,NOVCOD = b4_cod
            ,LFOG = lambda x : np.where(~np.isin(x['LFOG'], [0,1]), None, x['LFOG'])
            ,ROG = lambda x : np.where(~np.isin(x['ROG'], [0,1]), None, x['ROG'])
            ,TRSSF = lambda x : np.where(x['TRSSF'].isna(), None, x['TRSSF'])
        )
        .drop(columns = ['NOVPC1','NOVTRS1'])
    )
    
    non_rej_rc1 = df1[df1['REJ']!=1]
    rej_rc1 = df1[df1['REJ']==1].assign(
    	EPC = None
    	,NOVPC = None
    	,NOVTRS = None
    	,CADC = None
    	,TLC = None
    	,MAC = None
    	,MUC = None
    	,DDG = None
    	,SGC = None
    	,GCC = None
    	,LFOG = None
    	,ROG = None
    	,DDTRS = None
    	,TRSSC = None
    	,TRSSF = None
    	,DCKC = None
    	,HSSN = None
    	,GEOA = None
    	,TCROPA = None
    	,TNSSN = None
    	,TGEOASN = None
    	,RC61 = None
    	,RC62 = None
    	,RC63 = None
    )
    rc1 = pd.concat([non_rej_rc1, rej_rc1], ignore_index=True)
    
    if rc1['HSSN'].isna().sum() > 0:
        print('='*50)
        print('In RC1, HSSN is Blank. Please Correct...')
        print('='*50)
    if rc1[rc1['HSSN'] == 0].shape[0] > 0:
        print('='*50)
        print('In RC1, HSSN is Zero. Please Correct...')
        print('='*50)
    if rc1['GEOA'].isna().sum() > 0:
        print('='*50)
        print('In RC1, GEOA is Blank. Please Correct...')
        print('='*50)
    if rc1[rc1['GEOA'] == 0].shape[0] > 0:
        print('='*50)
        print('In RC1, GEOA is Zero. Please Correct...')
        print('='*50)
    
    
    rc1.to_excel(excel_writer='RC1237_clean.xlsx', sheet_name='RC1', index=False, engine='openpyxl')
    
    
    
    
    
    
    
    #get data from the table - RC2
    cols2 = ['RC', 'ST', 'YR', 'SESON', 'STAT', 'DIST', 'STRA', 'VILL', 'EPC', 
             'HSSN', 'TNSSN', 'CROP', 'VARC', 'ARSU', 'ARSI', 'ARPU', 'ARPI']
    
    sql_query2 = 'SELECT * FROM "Copy Of RC2"'
    df2 = pd.read_sql(sql_query2, conn)[cols2]
    df2 = (df2
        .drop_duplicates()
        .apply(pd.to_numeric, errors='coerce')
        .query("STAT == @state_cd and SESON == @season_cd")
        .assign(
            YR = year+1
            ,VARC = lambda x : np.where(x['CROP']==0, 0, x['VARC'])
            ,CROP = lambda x : np.where(x['VARC']==0, 0, x['CROP'])
        )
    )
    rc2 = df2.copy()
    
    with pd.ExcelWriter('RC1237_clean.xlsx', mode='a', engine='openpyxl') as w2:
        rc2.to_excel(w2, sheet_name='RC2', index=False)
    
    
    
    
    
    
    
    #get data from the table - RC3
    cols3 = ['RC', 'ST', 'YR', 'SESON', 'STAT', 'DIST', 'STRA',
             'VILL', 'SN', 'CROP', 'VARC', 'IRRC', 'ERC']
    
    sql_query3 = 'SELECT * FROM "Copy Of RC3"'
    df3 = pd.read_sql(sql_query3, conn)[cols3]
    df3 = (df3
        .apply(pd.to_numeric, errors='coerce')
        .query("STAT == @state_cd and SESON == @season_cd")
        .assign(
            YR = year+1
            ,VARC = lambda x : np.where(x['CROP']==0, 0, x['VARC'])
            ,CROP = lambda x : np.where(x['VARC']==0, 0, x['CROP'])
        )
    )
    
    #creating schedule unique id in RC1
    rc1['unique_id'] = rc1['STAT'].astype(str) + "_" + rc1['SESON'].astype(str) + "_" + rc1['ST'].astype(str) + "_" + rc1['STRA'].astype(str) + "_" + rc1['VILL'].astype(str)+ "_" + rc1['DIST'].astype(str)
    
    
    #creating schedule unique id in df3
    df3['unique_id'] = df3['STAT'].astype(str) + "_" + df3['SESON'].astype(str) + "_" + df3['ST'].astype(str) + "_" + df3['STRA'].astype(str) + "_" + df3['VILL'].astype(str)+ "_" + df3['DIST'].astype(str)
    
    #Left Join on df3 with rc1_epc1
    rc1_epc = rc1[['unique_id','EPC']]
    rc3 = pd.merge(df3, rc1_epc, how='left', left_on='unique_id', right_on='unique_id')
    rc3 = rc3[rc3['EPC'] == 1].drop(columns=['unique_id','EPC'])
    
    with pd.ExcelWriter('RC1237_clean.xlsx', mode='a', engine='openpyxl') as w3:
        rc3.to_excel(w3, sheet_name='RC3', index=False)
    
    
    
    
    
    conn.close()
    return rc1, rc2, rc3

In [44]:
year = 2024
state_cd = 33
season_cd = 1
crop_plan_c = {
    91 : 40,
    3 : 40,
    5 : 40,
    6 : 40,
    15 : 40,
    33 : 40,
    11 : 40,
    12 : 40,
    92 : 60,
    14 : 40
}

crop_plan_s = {
    91 : 40,
    3 : 40,
    5 : 40,
    6 : 40,
    15 : 40,
    33 : 40,
    11 : 40,
    12 : 40,
    92 : 60,
    14 : 40
}

crop_ccf = {
    91 : 0.9352* 0.6667* 0.4,
    3 : 0.6091* 1* 0.4,
    5 : 0.718* 1* 0.4,
    6 : 0.7863* 1* 0.4,
    15 : 0.7378* 1* 0.4,
    33 : 1* 1* 0.2,
    11 : 1* 1* 0.2,
    12 : 1* 0.3295* 0.2,
    92 : 0.9324* 0.6667* 0.4,
    14 : 1* 1* 0.4
}


d_ST = {1 : "Central", 2 : "State"}

d_state_nm = {
	1 : "Jammu & Kashmir",
	2 : "Himachal Pradesh",
	3 : "Punjab",
	5 : "Uttarakhand",
	6 : "Haryana",
	8 : "Rajasthan",
	9 : "Uttar Pradesh",
	10 : "Bihar",
	18 : "Assam",
	19 : "West Bengal",
	20 : "Jharkhand",
	21 : "Odisha",
	22 : "Chattisgarh",
	23 : "Madhya Pradesh",
	24 : "Gujarat",
	27 : "Maharashtra",
	28 : "Andhra Pradesh",
	29 : "Karnataka",
	32 : "Kerala",
	33 : "Tamil Nadu",
	34 : "Puducherry",
	36 : "Telangana"
}


d_state_season_nm = {
	1: {1: 'Kharif'},
	2: {1: 'Kharif', 4: 'Rabi'},
	3: {1: 'Kharif'},
	5: {1: 'Kharif', 4: 'Rabi'},
	6: {1: 'Kharif'},
	8: {1: 'Kharif'},
	9: {1: 'Kharif'},
	10: {2: 'Bhadai', 3: 'Aghani', 4: 'Rabi', 5: 'Summer'},
	18: {2: 'Early Kharif', 3: 'Late Kharif', 4: 'Rabi', 5: 'Summer'},
	19: {2: 'Early Kharif', 3: 'Late Kharif', 4: 'Rabi', 5: 'Summer'},
	20: {2: 'Bhadai', 3: 'Agahani'},
	21: {2: 'Autumn', 3: 'Winter', 5: 'Summer'},
	22: {1: 'Kharif', 4: 'Rabi'},
	23: {1: 'Kharif', 4: 'Rabi'},
	24: {1: 'Kharif', 4: 'Rabi', 5: 'Summer'},
	27: {1: 'Kharif'},
	28: {1: 'Kharif', 4: 'Rabi'},
	29: {1: 'Kharif'},
	32: {2: 'Autumn', 6: 'Whole Year'},
	33: {2: 'Phase-I', 3: 'Phase-II', 4: 'Phase-III'},
	34: {1: 'Kharif', 4: 'Rabi-I', 5: 'Rabi-II'},
	36: {1: 'Kharif', 4: 'Rabi'}
}

In [75]:
#set database file path
folder_path = Path('./software')
for file_path in folder_path.iterdir():
    if file_path.is_file():
        if 'NSSAS' in file_path.name:
            mdb_path = "./" + "/".join(str(file_path).split("\\"))

#create connection with mdb database
conn_str = (
    r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
    f'DBQ={mdb_path};'
)
conn = pyodbc.connect(conn_str)

#get data from the table - Sch20RC7
cols7 = ['RC', 'ST', 'YR', 'SESON', 'CROP', 'NEXP', 'NEXNR', 'CONFAC', 
         'STAT', 'DIST', 'VILL', 'RSV', 'EXPNO', 'STAGIN', 'REXML', 
         'CCE', 'CFC', 'RCSSN', 'E1', 'E2', 'E9', 'E10', 'E11', 'E12', 
         'ADH', 'QTYG', 'QTYP', 'E13', 'EQST', 'EQSB', 'EQSW', 'EQSP', 
         'E14', 'STYPE', 'E3', 'SVARC', 'SEEDR', 'E4', 'IRR', 'E5', 
         'IRRP', 'NIRR', 'FERTC', 'E6', 'QTN', 'CODN', 'QTP', 'CODP', 
         'QTK', 'CODK', 'MANUR', 'E7', 'PEST', 'E8', 'E15', 'REJ', 
         'CROPER', 'CROPCON', 'AFFECT']

sql_query7 = 'SELECT * FROM "Sch20RC7"'
rc7 = pd.read_sql(sql_query7, conn)[cols7]
rc7 = (rc7
    .drop_duplicates()
    .apply(pd.to_numeric, errors="coerce")
    .query("STAT == @state_cd and SESON == @season_cd")
    .assign(
        #adjusting cotton crop code
        YR = year+1
        ,CROP = lambda x : x['CROP'].replace({55: 12, 57: 56})
        ,REJ = lambda x : x['REJ'].apply(lambda y : 1 if y==1 else 0)
        ,RSV = lambda x: x['RSV'].where(x["RSV"].isin([0, 1, 2, 3, 9]), 0)
        ,REXML = lambda x: x["REXML"].where(
                    ~(x["STAGIN"].isin([2, 3]) & ~x["REXML"].isin([1, 2, 3, 4, 9])),
                    9
        )
        ,CFC = lambda x: x["CFC"].where(
                    ~(x["STAGIN"].isin([1, 2]) & ~x["CFC"].isin([1, 2, 3, 4])),
                    4
        )
        ,RCSSN = lambda x: x["RCSSN"].where(
            ~(x["STAGIN"].isin([1, 2]) & ~x["RCSSN"].isin([0, 1, 2, 3, 9])),
            0
        )
        ,EQST = lambda x: x["EQST"].where(
            ~(x["STAGIN"].isin([1, 2]) & x["EQST"].isna()),
            9
        )
        ,EQSB = lambda x: x["EQSB"].where(
            ~(x["STAGIN"].isin([1, 2]) & x["EQSB"].isna()),
            9
        )
        ,EQSW = lambda x: x["EQSW"].where(
            ~(x["STAGIN"].isin([1, 2]) & x["EQSW"].isna()),
            9
        )
        ,EQSP = lambda x: x["EQSP"].where(
            ~(x["STAGIN"].isin([1, 2]) & x["EQSP"].isna()),
            9
        )
        ,QTYG = lambda x: (
            x["QTYG"]
            .where(x["STAGIN"] != 2, None)
            .where(~(x["STAGIN"].eq(1) & x["CROPCON"].eq(4)), 0)
        )
        ,QTYP = lambda x: (
            x["QTYP"]
            .where(x["STAGIN"] != 2, None)
            .where(~(x["STAGIN"].eq(1) & x["CROPCON"].eq(4)), 0)
        )
        ,E13 = lambda x: x["E13"].where(x["STAGIN"] != 2, None)
        ,AFFECT = lambda x: x['AFFECT'].where(~((x['STAGIN'].isin([1,2])) & (x['CROPCON'].isin([1,2]))), 0)
        ,IRR = lambda x : np.where(x['NIRR']>0, 1, 0)
        ,CODN = lambda x : x['CODN'].where(~(x['QTN']>0), 1)
        ,CODP = lambda x : x['CODP'].where(~(x['QTP']>0), 1)
        ,CODK = lambda x : x['CODK'].where(~(x['QTK']>0), 1)
        ,FERTC = lambda x : np.where(((x['QTN']>0) | (x['QTP']>0) | (x['QTK']>0) | (x['CODN']==9) | (x['CODP']==9) | (x['CODK']==9)), 1, 0)
        
    )
)

rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
    (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
    ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'FERTC'] = 0

rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
    (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
    ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'QTN'] = np.nan

rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
    (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
    ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'QTP'] = np.nan

rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
    (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
    ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'QTK'] = np.nan

rc7.loc[~((rc7['CODN'].isin([1,8,9])) | (rc7['CODN'].isna())), 'CODN'] = np.nan
rc7.loc[~((rc7['CODP'].isin([1,8,9])) | (rc7['CODP'].isna())), 'CODP'] = np.nan
rc7.loc[~((rc7['CODK'].isin([1,8,9])) | (rc7['CODK'].isna())), 'CODK'] = np.nan

rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'FERTC'] = 0
rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'QTN'] = np.nan
rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'QTP'] = np.nan
rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'QTK'] = np.nan

rc7.loc[(rc7['FERTC']==0), 'QTN'] = np.nan
rc7.loc[(rc7['FERTC']==0), 'QTP'] = np.nan
rc7.loc[(rc7['FERTC']==0), 'QTK'] = np.nan
rc7.loc[(rc7['FERTC']==0), 'CODN'] = 8
rc7.loc[(rc7['FERTC']==0), 'CODP'] = 8
rc7.loc[(rc7['FERTC']==0), 'CODK'] = 8

lost_cols = ['CCE', 'CFC', 'RCSSN', 'E1', 'E2', 'E9', 'E10', 'E11', 'E12', 
             'ADH', 'QTYG', 'QTYP', 'E13', 'EQST', 'EQSB', 'EQSW', 'EQSP',
             'E14', 'STYPE', 'E3', 'SVARC', 'SEEDR', 'E4', 'IRR', 'E5',
             'IRRP', 'NIRR', 'FERTC', 'E6', 'QTN', 'CODN', 'QTP', 'CODP',
             'QTK', 'CODK', 'MANUR', 'E7', 'PEST', 'E8', 'E15','CROPER', 
             'CROPCON', 'AFFECT']

for col in lost_cols:
    rc7.loc[rc7['STAGIN']==3, col] = np.nan

for crop,plan in crop_plan_c.items():
    rc7.loc[((rc7['ST']==1) & (rc7['CROP']==crop)), 'NEXP'] = plan
    received = rc7[(rc7['ST']==1) & (rc7['CROP']==crop)].shape[0]
    rc7.loc[((rc7['ST']==1) & (rc7['CROP']==crop)), 'NEXR'] = received

for crop,plan in crop_plan_c.items():
    rc7.loc[((rc7['ST']==2) & (rc7['CROP']==crop)), 'NEXP'] = plan
    received = rc7[(rc7['ST']==1) & (rc7['CROP']==crop)].shape[0]
    rc7.loc[((rc7['ST']==2) & (rc7['CROP']==crop)), 'NEXR'] = received

for crop, ccf in crop_ccf.items():
    if ccf<1:
        ccf = np.floor(ccf*10000)
    rc7.loc[(rc7['CROP']==crop), 'CONFAC'] = ccf
    
rc7['NEXNR'] = rc7['NEXP'].sub(rc7['NEXR'], fill_value=0)

with pd.ExcelWriter('RC1237_clean.xlsx', mode='a', engine='openpyxl') as w7:
    rc7.to_excel(w7, sheet_name='RC7', index=False)

conn.close()

In [ ]:
def clean_rc7(year_int):


    #data to be provided
    year = 2024
    state_cd = 33
    season_cd = 1
    crop_plan_c = {
        91 : 40,
        3 : 40,
        5 : 40,
        6 : 40,
        15 : 40,
        33 : 40,
        11 : 40,
        12 : 40,
        92 : 60,
        14 : 40
    }
    
    crop_plan_s = {
        91 : 40,
        3 : 40,
        5 : 40,
        6 : 40,
        15 : 40,
        33 : 40,
        11 : 40,
        12 : 40,
        92 : 60,
        14 : 40
    }
    
    crop_ccf = {
        91 : 0.9352* 0.6667* 0.4,
        3 : 0.6091* 1* 0.4,
        5 : 0.718* 1* 0.4,
        6 : 0.7863* 1* 0.4,
        15 : 0.7378* 1* 0.4,
        33 : 1* 1* 0.2,
        11 : 1* 1* 0.2,
        12 : 1* 0.3295* 0.2,
        92 : 0.9324* 0.6667* 0.4,
        14 : 1* 1* 0.4
    }
    
    
    d_ST = {1 : "Central", 2 : "State"}


    
    #set database file path
    folder_path = Path('./software')
    for file_path in folder_path.iterdir():
        if file_path.is_file():
            if 'NSSAS' in file_path.name:
                mdb_path = "./" + "/".join(str(file_path).split("\\"))
    
    #create connection with mdb database
    conn_str = (
        r'DRIVER={Microsoft Access Driver (*.mdb, *.accdb)};'
        f'DBQ={mdb_path};'
    )
    conn = pyodbc.connect(conn_str)
    
    #get data from the table - Sch20RC7
    cols7 = ['RC', 'ST', 'YR', 'SESON', 'CROP', 'NEXP', 'NEXNR', 'CONFAC', 
             'STAT', 'DIST', 'VILL', 'RSV', 'EXPNO', 'STAGIN', 'REXML', 
             'CCE', 'CFC', 'RCSSN', 'E1', 'E2', 'E9', 'E10', 'E11', 'E12', 
             'ADH', 'QTYG', 'QTYP', 'E13', 'EQST', 'EQSB', 'EQSW', 'EQSP', 
             'E14', 'STYPE', 'E3', 'SVARC', 'SEEDR', 'E4', 'IRR', 'E5', 
             'IRRP', 'NIRR', 'FERTC', 'E6', 'QTN', 'CODN', 'QTP', 'CODP', 
             'QTK', 'CODK', 'MANUR', 'E7', 'PEST', 'E8', 'E15', 'REJ', 
             'CROPER', 'CROPCON', 'AFFECT']
    
    sql_query7 = 'SELECT * FROM "Sch20RC7"'
    rc7 = pd.read_sql(sql_query7, conn)[cols7]
    rc7 = (rc7
        .drop_duplicates()
        .apply(pd.to_numeric, errors="coerce")
        .query("STAT == @state_cd and SESON == @season_cd")
        .assign(
            #adjusting cotton crop code
            YR = year+1
            ,CROP = lambda x : x['CROP'].replace({55: 12, 57: 56})
            ,REJ = lambda x : x['REJ'].apply(lambda y : 1 if y==1 else 0)
            ,RSV = lambda x: x['RSV'].where(x["RSV"].isin([0, 1, 2, 3, 9]), 0)
            ,REXML = lambda x: x["REXML"].where(
                        ~(x["STAGIN"].isin([2, 3]) & ~x["REXML"].isin([1, 2, 3, 4, 9])),
                        9
            )
            ,CFC = lambda x: x["CFC"].where(
                        ~(x["STAGIN"].isin([1, 2]) & ~x["CFC"].isin([1, 2, 3, 4])),
                        4
            )
            ,RCSSN = lambda x: x["RCSSN"].where(
                ~(x["STAGIN"].isin([1, 2]) & ~x["RCSSN"].isin([0, 1, 2, 3, 9])),
                0
            )
            ,EQST = lambda x: x["EQST"].where(
                ~(x["STAGIN"].isin([1, 2]) & x["EQST"].isna()),
                9
            )
            ,EQSB = lambda x: x["EQSB"].where(
                ~(x["STAGIN"].isin([1, 2]) & x["EQSB"].isna()),
                9
            )
            ,EQSW = lambda x: x["EQSW"].where(
                ~(x["STAGIN"].isin([1, 2]) & x["EQSW"].isna()),
                9
            )
            ,EQSP = lambda x: x["EQSP"].where(
                ~(x["STAGIN"].isin([1, 2]) & x["EQSP"].isna()),
                9
            )
            ,QTYG = lambda x: (
                x["QTYG"]
                .where(x["STAGIN"] != 2, None)
                .where(~(x["STAGIN"].eq(1) & x["CROPCON"].eq(4)), 0)
            )
            ,QTYP = lambda x: (
                x["QTYP"]
                .where(x["STAGIN"] != 2, None)
                .where(~(x["STAGIN"].eq(1) & x["CROPCON"].eq(4)), 0)
            )
            ,E13 = lambda x: x["E13"].where(x["STAGIN"] != 2, None)
            ,AFFECT = lambda x: x['AFFECT'].where(~((x['STAGIN'].isin([1,2])) & (x['CROPCON'].isin([1,2]))), 0)
            ,IRR = lambda x : np.where(x['NIRR']>0, 1, 0)
            ,CODN = lambda x : x['CODN'].where(~(x['QTN']>0), 1)
            ,CODP = lambda x : x['CODP'].where(~(x['QTP']>0), 1)
            ,CODK = lambda x : x['CODK'].where(~(x['QTK']>0), 1)
            ,FERTC = lambda x : np.where(((x['QTN']>0) | (x['QTP']>0) | (x['QTK']>0) | (x['CODN']==9) | (x['CODP']==9) | (x['CODK']==9)), 1, 0)
            
        )
    )
    
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
        (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
        ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'FERTC'] = 0
    
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
        (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
        ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'QTN'] = np.nan
    
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
        (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
        ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'QTP'] = np.nan
    
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN']==8) & (rc7['CODP']==8) & 
        (rc7['CODK']==8) & ((rc7['QTN']==0) | rc7['QTN'].isna()) & 
        ((rc7['QTP']==0) | rc7['QTP'].isna()) & ((rc7['QTK']==0) | rc7['QTK'].isna())), 'QTK'] = np.nan
    
    rc7.loc[~((rc7['CODN'].isin([1,8,9])) | (rc7['CODN'].isna())), 'CODN'] = np.nan
    rc7.loc[~((rc7['CODP'].isin([1,8,9])) | (rc7['CODP'].isna())), 'CODP'] = np.nan
    rc7.loc[~((rc7['CODK'].isin([1,8,9])) | (rc7['CODK'].isna())), 'CODK'] = np.nan
    
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'FERTC'] = 0
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'QTN'] = np.nan
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'QTP'] = np.nan
    rc7.loc[((rc7['STAGIN'].isin([1,2])) & (rc7['CODN'].isna()) & (rc7['CODP'].isna()) & (rc7['CODK'].isna())), 'QTK'] = np.nan
    
    rc7.loc[(rc7['FERTC']==0), 'QTN'] = np.nan
    rc7.loc[(rc7['FERTC']==0), 'QTP'] = np.nan
    rc7.loc[(rc7['FERTC']==0), 'QTK'] = np.nan
    rc7.loc[(rc7['FERTC']==0), 'CODN'] = 8
    rc7.loc[(rc7['FERTC']==0), 'CODP'] = 8
    rc7.loc[(rc7['FERTC']==0), 'CODK'] = 8
    
    lost_cols = ['CCE', 'CFC', 'RCSSN', 'E1', 'E2', 'E9', 'E10', 'E11', 'E12', 
                 'ADH', 'QTYG', 'QTYP', 'E13', 'EQST', 'EQSB', 'EQSW', 'EQSP',
                 'E14', 'STYPE', 'E3', 'SVARC', 'SEEDR', 'E4', 'IRR', 'E5',
                 'IRRP', 'NIRR', 'FERTC', 'E6', 'QTN', 'CODN', 'QTP', 'CODP',
                 'QTK', 'CODK', 'MANUR', 'E7', 'PEST', 'E8', 'E15','CROPER', 
                 'CROPCON', 'AFFECT']
    
    for col in lost_cols:
        rc7.loc[rc7['STAGIN']==3, col] = np.nan
    
    for crop,plan in crop_plan_c.items():
        rc7.loc[((rc7['ST']==1) & (rc7['CROP']==crop)), 'NEXP'] = plan
        received = rc7[(rc7['ST']==1) & (rc7['CROP']==crop)].shape[0]
        rc7.loc[((rc7['ST']==1) & (rc7['CROP']==crop)), 'NEXR'] = received
    
    for crop,plan in crop_plan_c.items():
        rc7.loc[((rc7['ST']==2) & (rc7['CROP']==crop)), 'NEXP'] = plan
        received = rc7[(rc7['ST']==1) & (rc7['CROP']==crop)].shape[0]
        rc7.loc[((rc7['ST']==2) & (rc7['CROP']==crop)), 'NEXR'] = received
    
    for crop, ccf in crop_ccf.items():
        if ccf<1:
            ccf = np.floor(ccf*10000)
        rc7.loc[(rc7['CROP']==crop), 'CONFAC'] = ccf
        
    rc7['NEXNR'] = rc7['NEXP'].sub(rc7['NEXR'], fill_value=0)
    
    with pd.ExcelWriter('RC1237_clean.xlsx', mode='a', engine='openpyxl') as w7:
        rc7.to_excel(w7, sheet_name='RC7', index=False)
    
    conn.close()
    return rc7

In [3]:
def granularize(rc1, rc2, rc3, rc4):
    pass

In [4]:
def generate_annex(grc1, grc2, grc3, grc4):
    pass

In [ ]:
if __name__ == "__main__":
    #read & clean RCs
    year_int = 2024            #2024 for A.Y. 2024-25
    season_cd = [2,3]              #1-Kharif, 4-Rabi
    rc1_novp = {
        1 : [0,0],                     ##season_1 planning [cen, state]
        2 : [260, 260],                ##season_2 planning [cen, state]
        3 : [260, 260],                ##season_3 planning [cen, state]
        4 : [260, 260],                ##season_4 planning [cen, state]
        5 : [0,0]                      ##season_5 planning [cen, state]
    }
    rc1_novcod = {
        1 : [0,0],                     ##season_1 NOVCOD [cen, state]
        2 : [260, 260],                ##season_2 NOVCOD [cen, state]
        3 : [260, 260],                ##season_3 NOVCOD [cen, state]
        4 : [260, 260],                ##season_4 NOVCOD [cen, state]
        5 : [0,0]                      ##season_5 NOVCOD [cen, state]
    }

    rc1, rc2, rc3 = clean_rc123(year_int, season_cd, rc1_novp, rc1_novcod)
    rc7 = clean_rc7(year_int, season_cd)

    #Generate Annexure Data
    generate_annex(rc1, rc2, rc3, rc7)